# OpenCSI full package → Google Drive

Notebook này chuyển dữ liệu **server-to-server**: GitHub Release/Figshare → Google Drive. Máy cá nhân không phải tải file 2 GB.

Thư mục đích đã được cấu hình sẵn theo Drive folder mà thầy cung cấp. Chạy lần lượt hai cell bên dưới, cấp quyền Google Drive khi Colab hỏi.


In [ ]:
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
drive = build('drive', 'v3')

ROOT_FOLDER_ID = '11TXh4knlWAZE1L7NHwemX421fp5dRZPK'
RAW_FOLDER_ID = '1VQQtjsBuj_kjqbYa1Slgc7OV75Mlw7bg'
PACKAGES_FOLDER_ID = '1lOioNSWefKK2RTmhkGFzPgeRJ_9_MYFV'
CHECKSUMS_FOLDER_ID = '1FxxPxaUlq8ThJKO3dPsAMj040Mb_HRbn'
PAPER_FOLDER_ID = '1DxUCcFf9r7zUx0vEcuCwpaTtJbenZ1nG'
SOURCE_RESULTS_FOLDER_ID = '18HZV77QpQUhX9gxHlt7Ohps0Gyu5CLfp'
RELEASE_TAG = 'opencsi-full-drive-transfer-v1'
RELEASE_BASE = f'https://github.com/nhan-four/Backdoor_Channel_Estimation/releases/download/{RELEASE_TAG}'
print('Google Drive authenticated. Destination root:', ROOT_FOLDER_ID)


In [ ]:
import hashlib, json, os, pathlib, time, requests
from urllib.parse import quote

WORK = pathlib.Path('/content/opencsi_drive_transfer')
WORK.mkdir(parents=True, exist_ok=True)
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'OpenCSI-Repro-Transfer/1.0'})

def sha256_file(path, chunk=8*1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def download(url, target, expected_size=None, expected_sha256=None):
    target = pathlib.Path(target)
    tmp = target.with_suffix(target.suffix + '.part')
    done = tmp.stat().st_size if tmp.exists() else 0
    headers = {'Range': f'bytes={done}-'} if done else {}
    mode = 'ab' if done else 'wb'
    with SESSION.get(url, stream=True, timeout=(30, 300), headers=headers, allow_redirects=True) as r:
        if done and r.status_code == 200:
            done = 0; mode = 'wb'
        r.raise_for_status()
        total = expected_size or (done + int(r.headers.get('content-length', 0)))
        last = time.time()
        with open(tmp, mode) as f:
            for chunk in r.iter_content(chunk_size=8*1024*1024):
                if not chunk: continue
                f.write(chunk); done += len(chunk)
                if time.time() - last > 5:
                    pct = 100.0 * done / total if total else 0
                    print(f'  {target.name}: {done/1e6:.1f}/{total/1e6:.1f} MB ({pct:.1f}%)')
                    last = time.time()
    if expected_size is not None and tmp.stat().st_size != expected_size:
        raise RuntimeError(f'Size mismatch for {target.name}: {tmp.stat().st_size} != {expected_size}')
    if expected_sha256 is not None:
        got = sha256_file(tmp)
        if got.lower() != expected_sha256.lower():
            raise RuntimeError(f'SHA256 mismatch for {target.name}: {got} != {expected_sha256}')
    tmp.replace(target)
    return target

def drive_find(parent_id, name):
    safe = name.replace("\'", "\\\'")
    q = f"'{parent_id}' in parents and name = '{safe}' and trashed = false"
    res = drive.files().list(q=q, fields='files(id,name,size,md5Checksum,webViewLink)', pageSize=10).execute()
    return res.get('files', [])

def drive_upload(path, parent_id, mime='application/octet-stream'):
    path = pathlib.Path(path)
    existing = drive_find(parent_id, path.name)
    for item in existing:
        if int(item.get('size', -1)) == path.stat().st_size:
            print('SKIP existing same-size:', path.name, item.get('webViewLink', item['id']))
            return item
    media = MediaFileUpload(str(path), mimetype=mime, resumable=True, chunksize=100*1024*1024)
    req = drive.files().create(body={'name': path.name, 'parents': [parent_id]}, media_body=media, fields='id,name,size,md5Checksum,webViewLink')
    response = None
    while response is None:
        status, response = req.next_chunk(num_retries=8)
        if status:
            print(f'  Drive {path.name}: {status.progress()*100:.1f}%')
    print('UPLOADED:', response['name'], response.get('webViewLink', response['id']))
    return response

manifest_url = f'{RELEASE_BASE}/release_manifest.json'
print('Reading release manifest:', manifest_url)
manifest = SESSION.get(manifest_url, timeout=60).json()

# Download and reconstruct the 2.03 GB upstream raw archive from four release parts.
raw_parts = []
for item in manifest['raw_parts']:
    p = download(f"{RELEASE_BASE}/{quote(item['name'])}", WORK/item['name'], item['size_bytes'], item['sha256'])
    raw_parts.append(p)
raw_archive = WORK / manifest['raw_archive']['name']
with open(raw_archive.with_suffix('.zip.part'), 'wb') as out:
    for p in raw_parts:
        with open(p, 'rb') as src:
            while True:
                b = src.read(16*1024*1024)
                if not b: break
                out.write(b)
if raw_archive.with_suffix('.zip.part').stat().st_size != manifest['raw_archive']['size_bytes']:
    raise RuntimeError('Reconstructed raw archive size mismatch')
if sha256_file(raw_archive.with_suffix('.zip.part')) != manifest['raw_archive']['sha256']:
    raise RuntimeError('Reconstructed raw archive SHA256 mismatch')
raw_archive.with_suffix('.zip.part').replace(raw_archive)
print('Raw archive verified:', raw_archive, raw_archive.stat().st_size)
drive_upload(raw_archive, RAW_FOLDER_ID, 'application/zip')

destinations = {
  'opencsi-compact-v8-measured.zip': (SOURCE_RESULTS_FOLDER_ID, 'application/zip'),
  'OPENCSI_FROZEN_RUNS_CHECKPOINTS.zip': (SOURCE_RESULTS_FOLDER_ID, 'application/zip'),
  'OPENCSI_FULL_PAPER_SRC_RESULTS_DOCKER.zip': (PAPER_FOLDER_ID, 'application/zip'),
  'SHA256SUMS.txt': (CHECKSUMS_FOLDER_ID, 'text/plain'),
  'release_manifest.json': (CHECKSUMS_FOLDER_ID, 'application/json'),
}
for item in manifest['package_assets']:
    p = download(f"{RELEASE_BASE}/{quote(item['name'])}", WORK/item['name'], item['size_bytes'], item['sha256'])
    parent, mime = destinations[item['name']]
    drive_upload(p, parent, mime)

# Upload manifest itself, even if it was read from memory.
manifest_path = WORK/'release_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')
drive_upload(manifest_path, CHECKSUMS_FOLDER_ID, 'application/json')
print('\nTRANSFER COMPLETE. All files were verified before Drive upload.')
